# Selective facts parser — train and evaluate
Use **GPU T4 x2**, **Internet on**, attach only your original **1.5B adapter** for the control comparison. No 3B adapter, GitHub push or earlier result ZIP is needed.

The job trains a **new** adapter from the pinned 1.5B base. It uses selective facts for supported statistic-fill requests, with the request last. Replay examples and new scoped examples use this same input format.

A two-step infrastructure smoke job runs first, followed by a fresh full two-epoch job. Each epoch is scored by free generation on 80 synthetic open-development tasks and 20 existing development tasks. The original adapter uses its original input format as the control. These scores are diagnostic; synthetic additions are automatically checked drafts without independent review. Training validation sources are separate from training sources. No held-out selection inference or test scoring is run.

Download `selective_facts_1_5b.zip` and the executed notebook. A valid completed run may still fail quality screening. Inspect its result before using the adapter in Stage 2.

In [ ]:
import base64,hashlib,json,os,subprocess,sys,zlib
from pathlib import Path
WORK=Path('/kaggle/working')
WORK.mkdir(parents=True,exist_ok=True)
PAYLOAD_SHA='94b61efe7bc8ceb2ee106eba081d76d59a7d40b35ac8500e9a5b49cd54d26e67'
PAYLOAD=(
'...')
raw=zlib.decompress(base64.b64decode(PAYLOAD))
assert hashlib.sha256(raw).hexdigest()==PAYLOAD_SHA
ROOT=WORK/('selective_facts_source_'+PAYLOAD_SHA[:12])
for name,value in json.loads(raw).items():
    relative=Path(name)
    assert not relative.is_absolute() and '..' not in relative.parts
    path=ROOT/relative;path.parent.mkdir(parents=True,exist_ok=True)
    content=base64.b64decode(value)
    if path.exists():assert path.read_bytes()==content
    else:path.write_bytes(content)
PACKAGES=WORK/'selective_facts_packages'
subprocess.run([sys.executable,'-m','pip','install','-q','--upgrade','--target',str(PACKAGES),'-r',str(ROOT/'requirements.txt')],check=True)
ENV=dict(os.environ,PYTHONPATH=os.pathsep.join(map(str,[ROOT,ROOT/'src',PACKAGES])),PYTHONUNBUFFERED='1',TOKENIZERS_PARALLELISM='false')
print('Verified embedded source and data:',PAYLOAD_SHA)


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==5.20.2','peft==0.21.2','bitsandbytes==0.50.2','accelerate==1.15.0','tokenizers==0.23.3','safetensors==0.8.0','huggingface-hub==1.33.0'],check=True)
subprocess.run([sys.executable,'-m','eval.stage2_independent_v2.environment','--prepare'],cwd=ROOT,env=ENV,check=True)
subprocess.run([sys.executable,'-m','finetune.selective_facts.preflight'],cwd=ROOT,env=ENV,check=True)
subprocess.run([sys.executable,'-c',"import torch; assert torch.__version__=='2.11.0+cu130'; assert torch.cuda.device_count()==2; assert all('T4' in torch.cuda.get_device_name(i) for i in (0,1)); print('T4 x2 preflight passed')"],cwd=ROOT,env=ENV,check=True)
spec=json.loads((ROOT/'eval/stage2_independent_v2/adapter.json').read_text())
matches=[]
for config in Path('/kaggle/input').rglob('adapter_config.json'):
    try:
        if json.loads(config.read_text()).get('base_model_name_or_path')==spec['model']:
            if all((config.parent/n).is_file() and hashlib.sha256((config.parent/n).read_bytes()).hexdigest()==sha for n,sha in spec['sha256'].items()):matches.append(config.parent)
    except (ValueError,OSError):pass
assert len(matches)==1, f'Attach exactly one original 1.5B adapter dataset; found {matches}'
BASELINE_ADAPTER=matches[0]
print('Original control adapter:',BASELINE_ADAPTER)


In [ ]:
def execute(smoke):
    out=WORK/('selective_facts_1_5b_smoke' if smoke else 'selective_facts_1_5b')
    assert not out.exists(), 'Start a fresh session or choose a new output directory.'
    command=[sys.executable,'-m','finetune.selective_facts.train','--data',str(ROOT/'finetune/data/selective_facts_v1'),'--out',str(out),'--baseline-adapter',str(BASELINE_ADAPTER),'--packages',str(PACKAGES)]
    if smoke:command+=['--smoke']
    # Training sees only physical GPU 0. Semantic subprocesses explicitly use physical GPU 1.
    training_env=dict(ENV,CUDA_VISIBLE_DEVICES='0')
    log=WORK/(out.name+'.log')
    with log.open('w') as f:
        process=subprocess.Popen(command,cwd=ROOT,env=training_env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            print(line,end='');f.write(line);f.flush()
        code=process.wait()
    assert code==0, f'Job failed. Download {log} and the partial ZIP for diagnosis.'
    assert json.loads((out/'completion.json').read_text())['complete']
    return out
SMOKE=execute(True)
print('Infrastructure smoke completed. Starting full fresh training in the next cell.')


In [ ]:
OUT=execute(False)
selection=json.loads((OUT/'selection.json').read_text())
print('Passed development screening:',selection['eligible'])
print('Original baseline:',json.dumps(selection['baseline'],indent=2))
print('New selected candidate:',json.dumps(selection['selected'],indent=2))
from IPython.display import FileLink,display
display(FileLink(str(OUT.with_suffix('.zip'))))
print('COMPLETE: training, full validation loss and semantic checkpoint evaluation.')
if not selection['eligible']:print('No checkpoint met the declared screening rule. Share the results for analysis before Stage 2.')
